In [1]:
# Will They Pay? — Production Model Training & Evaluation
# Loads the flat training table from `build_training_data_prod.ipynb`, trains and compares
# baseline models (Logistic Regression, Random Forest, XGBoost), applies defensive evaluation
# layers (SMOTE, PR threshold tuning, lift metrics), and batch-scores open invoices.

# **ML Workflow Steps:**
# 1. Load production training data
# 2. Define features and target
# 3. Split data (80/20)
# 4. Establish baselines (majority class, prior probability)
# 5. Train and compare models (with SMOTE)
# 6. Evaluate performance (ROC-AUC, PR curve, lift metrics)
# 6d. Synthetic feature impact analysis (A/B comparison)
# 7. Feature importance
# 8. Save model artifact
# 9. Batch score open invoices
# 10. Save scored predictions

## Step 1: Load Production Training Data

In [2]:
import pandas as pd
import numpy as np

training_df = pd.read_csv('../../data/processed/training_data_prod.csv')

print(f"Loaded training data: {training_df.shape[0]:,} rows x {training_df.shape[1]} columns")
print(f"\nTarget distribution (paid_within_30_days):")
print(f"   Paid within 30d (1): {training_df['paid_within_30_days'].sum():,} ({training_df['paid_within_30_days'].mean()*100:.1f}%)")
print(f"   Late/unpaid     (0): {(training_df['paid_within_30_days'] == 0).sum():,} ({(1 - training_df['paid_within_30_days'].mean())*100:.1f}%)")
training_df.head()

Loaded training data: 24,355 rows x 21 columns

Target distribution (paid_within_30_days):
   Paid within 30d (1): 19,103 (78.4%)
   Late/unpaid     (0): 5,252 (21.6%)


,personid,postal_code,amount,created_day_of_week,created_day_of_month,created_hour,payment_origin_mode,surchargingenabled,is_guardian,account_age_days,...,is_new_patient,average_days_to_pay,appointment_reliability_score,median_household_income,poverty_rate_pct,average_household_size,bachelors_or_higher_pct,unemployment_rate_pct,status,paid_within_30_days
0,00c51f14-97b7-551a-a5a2-6a453fd71a5c,77489.0,52613,3,20,16,1.0,0,0,0.0,...,0,0.0,1.00,74097.0,9.9,2.91,22.3,4.2,4,1
1,00c51f14-97b7-551a-a5a2-6a453fd71a5c,77489.0,48600,3,16,15,1.0,0,0,237.0,...,0,0.0,1.00,74097.0,9.9,2.91,22.3,4.2,4,1
2,00c51f14-97b7-551a-a5a2-6a453fd71a5c,77489.0,221100,1,7,19,1.0,0,0,411.0,...,0,0.0,1.00,74097.0,9.9,2.91,22.3,4.2,4,1
3,11af2c63-9a57-556a-91e7-c157840adf45,88081.0,106520,3,7,16,1.0,0,1,0.0,...,0,9.0,0.75,46102.0,28.8,3.49,9.1,4.0,6,0
4,11af2c63-9a57-556a-91e7-c157840adf45,88081.0,19000,6,7,19,1.0,0,1,31.0,...,0,9.0,0.75,46102.0,28.8,3.49,9.1,4.0,6,0


## Step 2: Define Features and Target

Production feature set uses `payment_origin_mode` (per-person mode of payment channel)
instead of per-invoice `origin`, since `origin` lives on `payment_log` not on `invoices`.

**Feature Flags:** Toggle `INCLUDE_SYNTHETIC_APPOINTMENTS` and `INCLUDE_SYNTHETIC_PERSON_FEATURES`
to test model performance with and without synthetic data.

In [3]:
# =============================================================================
# FEATURE FLAGS: Toggle synthetic features on/off for A/B comparison
# =============================================================================
# Set to False to exclude synthetic features and measure their impact on model performance.
# When False, the notebook will train models without these features and compare results.

INCLUDE_SYNTHETIC_APPOINTMENTS = True   # appointment_reliability_score (from dummy calendar events)
INCLUDE_SYNTHETIC_PERSON_FEATURES = True  # is_guardian, account_age_days, tenure_months (from dummy persons)

# =============================================================================
# FEATURE DEFINITIONS BY SOURCE
# =============================================================================

# Real production features (from payments.invoices, payment_log, census)
REAL_FEATURES = [
    # Invoice features
    'amount',
    'created_day_of_week',
    'created_day_of_month',
    'created_hour',
    'payment_origin_mode',           # per-person mode; origin is on payment_log, not invoices
    'surchargingenabled',
    # Derived behavioral features (from real payment_log data)
    'average_days_to_pay',
    'is_new_patient',                # derived from payment history (no prior payments)
    # Census features (real public data)
    'median_household_income',
    'poverty_rate_pct',
    'average_household_size',
    'bachelors_or_higher_pct',
    'unemployment_rate_pct',
]

# Synthetic person features (from dummy public.person records)
SYNTHETIC_PERSON_FEATURES = [
    'is_guardian',
    'account_age_days',
    'tenure_months',
]

# Synthetic appointment features (from dummy scheduler.calendars_events)
SYNTHETIC_APPOINTMENT_FEATURES = [
    'appointment_reliability_score',
]

# =============================================================================
# BUILD FEATURE LIST BASED ON FLAGS
# =============================================================================

feature_columns = REAL_FEATURES.copy()
if INCLUDE_SYNTHETIC_PERSON_FEATURES:
    feature_columns.extend(SYNTHETIC_PERSON_FEATURES)
if INCLUDE_SYNTHETIC_APPOINTMENTS:
    feature_columns.extend(SYNTHETIC_APPOINTMENT_FEATURES)

print('='*60)
print(' FEATURE CONFIGURATION')
print('='*60)
print(f'  Real features:                 {len(REAL_FEATURES)}')
print(f'  Synthetic person features:     {len(SYNTHETIC_PERSON_FEATURES)} ({"INCLUDED" if INCLUDE_SYNTHETIC_PERSON_FEATURES else "EXCLUDED"})')
print(f'  Synthetic appointment features: {len(SYNTHETIC_APPOINTMENT_FEATURES)} ({"INCLUDED" if INCLUDE_SYNTHETIC_APPOINTMENTS else "EXCLUDED"})')
print(f'  Total features:                {len(feature_columns)}')
print()
if not INCLUDE_SYNTHETIC_APPOINTMENTS:
    print('  ⚠️  Synthetic appointment features EXCLUDED for comparison run')
if not INCLUDE_SYNTHETIC_PERSON_FEATURES:
    print('  ⚠️  Synthetic person features EXCLUDED for comparison run')
print()

# Drop rows with NaN in feature columns.
# These are invoices for persons with no matching dummy person/address record
# (distinct from the timestamp issue, which is now fixed in build_training_data_prod.ipynb).
nan_mask = training_df[feature_columns].notna().all(axis=1)
nan_dropped = (~nan_mask).sum()
if nan_dropped > 0:
    print(f"Dropping {nan_dropped} rows with NaN in feature columns (no matching dummy person record)")
    training_df = training_df[nan_mask].reset_index(drop=True)

X = training_df[feature_columns]
y = training_df['paid_within_30_days']

print(f"Features: {X.shape[1]} columns")
print(f"Target: {y.shape[0]:,} rows")
print(f"\nNull check:")
print(X.isnull().sum().to_string())

 FEATURE CONFIGURATION
  Real features:                 13
  Synthetic person features:     3 (INCLUDED)
  Synthetic appointment features: 1 (INCLUDED)
  Total features:                17


Dropping 4 rows with NaN in feature columns (no matching dummy person record)
Features: 17 columns
Target: 24,351 rows

Null check:
amount                           0
created_day_of_week              0
created_day_of_month             0
created_hour                     0
payment_origin_mode              0
surchargingenabled               0
average_days_to_pay              0
is_new_patient                   0
median_household_income          0
poverty_rate_pct                 0
average_household_size           0
bachelors_or_higher_pct          0
unemployment_rate_pct            0
is_guardian                      0
account_age_days                 0
tenure_months                    0
appointment_reliability_score    0


## Diagnosis: Root Cause

**The data is fine.** The 1,116 dropped rows are valid timestamps — they just lack sub-second precision:

| Format | Count | Example |
|---|---|---|
| With microseconds | 23,239 | `2022-12-15 17:16:46.426343+00:00` |
| **Without sub-seconds** | **1,116** | `2021-02-24 19:07:39+00:00` |

`pd.to_datetime(..., utc=True)` with no `format` argument defaults to inferring a single format from the first rows. Once it locks on the microsecond format (`%Y-%m-%d %H:%M:%S.%f%z`), it silently produces `NaT` for the 1,116 rows that are missing the `.%f` part.

**Fix:** Pass `format='mixed'` to `pd.to_datetime` in `build_training_data_prod.ipynb` — this parses each row individually and handles both formats. With `format='mixed'`, **0 NaT** are produced.

This should be fixed in `build_training_data_prod.ipynb` and the training data regenerated to recover the 1,116 rows (~4.6% of data).

## Step 3: Split Data (Train / Test)

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train set: {X_train.shape[0]:,} rows")
print(f"Test set:  {X_test.shape[0]:,} rows")
print(f"\nTrain target distribution: {y_train.value_counts().to_dict()}")
print(f"Test target distribution:  {y_test.value_counts().to_dict()}")

Train set: 19,480 rows
Test set:  4,871 rows

Train target distribution: {1: 15279, 0: 4201}
Test target distribution:  {1: 3820, 0: 1051}


## Step 4: Establish Baselines

Dumb baselines the real model must beat:
- **Majority class**: predict everyone pays
- **Prior probability**: predict the overall pay rate for everyone

In [5]:
from sklearn.metrics import (
    accuracy_score, brier_score_loss, f1_score, log_loss,
    precision_score, recall_score, roc_auc_score, classification_report,
    confusion_matrix, precision_recall_curve
)


def classification_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }


y_test_arr = y_test.to_numpy()
paid_rate = y_test_arr.mean()

majority_metrics = classification_metrics(y_test_arr, np.ones(len(y_test_arr), dtype=int))
prior_probs = np.full(len(y_test_arr), paid_rate)

print("=== Majority Class Baseline ===")
for name, value in majority_metrics.items():
    print(f"  {name.title():<10} {value:.3f}")

print("\n=== Prior Probability Baseline ===")
print(f"  Paid rate   {paid_rate:.3f}")
print(f"  Brier       {brier_score_loss(y_test_arr, prior_probs):.4f}")
print(f"  Log-loss    {log_loss(y_test_arr, prior_probs):.4f}")

=== Majority Class Baseline ===
  Accuracy   0.784
  Precision  0.784
  Recall     1.000
  F1         0.879

=== Prior Probability Baseline ===
  Paid rate   0.784
  Brier       0.1692
  Log-loss    0.5215


## Step 5: Train and Compare Models

Trains three models with SMOTE resampling on training data:
1. **Logistic Regression** — fast, interpretable
2. **Random Forest** — handles nonlinear interactions
3. **XGBoost** — often strongest performer

SMOTE generates synthetic minority-class examples to balance training data
(applied only to training fold, never test data).

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

models = {
    'Logistic Regression': ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'))
    ]),
    'Random Forest': ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('clf', RandomForestClassifier(
            n_estimators=200, random_state=42, class_weight='balanced'
        ))
    ]),
    'XGBoost': ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('clf', XGBClassifier(
            n_estimators=200, random_state=42, eval_metric='logloss',
            scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum()
        ))
    ]),
}

trained_models = {}
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f"   Done.")

print(f"\n\u2705 All {len(trained_models)} models trained (with SMOTE resampling).")

Training Logistic Regression...
   Done.
Training Random Forest...
   Done.
Training XGBoost...
   Done.

✅ All 3 models trained (with SMOTE resampling).


## Step 6: Evaluate Performance

Compares models by ROC-AUC, accuracy, precision/recall.
Prioritizes precision on the "unlikely to pay" class per PRD.

In [7]:
results = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    brier = brier_score_loss(y_test, y_proba)

    results.append({'Model': name, 'Accuracy': acc, 'ROC-AUC': auc, 'Brier': brier})

    print(f"\n{'='*60}")
    print(f" {name}")
    print(f"{'='*60}")
    print(f" Accuracy: {acc:.4f}")
    print(f" ROC-AUC:  {auc:.4f}")
    print(f" Brier:    {brier:.4f}")
    print(f"\n Classification Report:")
    print(classification_report(y_test, y_pred, target_names=['Late/Unpaid', 'Paid']))
    print(f" Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

# Summary comparison
results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
print(f"\n\n{'='*60}")
print(" MODEL COMPARISON (sorted by ROC-AUC)")
print(f"{'='*60}")
print(results_df.to_string(index=False))

best_model_name = results_df.iloc[0]['Model']
print(f"\n\u2705 Best model: {best_model_name} (AUC={results_df.iloc[0]['ROC-AUC']:.4f})")


 Logistic Regression
 Accuracy: 0.7647
 ROC-AUC:  0.8304
 Brier:    0.1598

 Classification Report:
              precision    recall  f1-score   support

 Late/Unpaid       0.47      0.70      0.56      1051
        Paid       0.91      0.78      0.84      3820

    accuracy                           0.76      4871
   macro avg       0.69      0.74      0.70      4871
weighted avg       0.81      0.76      0.78      4871

 Confusion Matrix:
[[ 739  312]
 [ 834 2986]]

 Random Forest
 Accuracy: 0.8838
 ROC-AUC:  0.9291
 Brier:    0.0828

 Classification Report:
              precision    recall  f1-score   support

 Late/Unpaid       0.75      0.69      0.72      1051
        Paid       0.92      0.94      0.93      3820

    accuracy                           0.88      4871
   macro avg       0.83      0.81      0.82      4871
weighted avg       0.88      0.88      0.88      4871

 Confusion Matrix:
[[ 723  328]
 [ 238 3582]]

 XGBoost
 Accuracy: 0.8926
 ROC-AUC:  0.9469
 Brier:    0

## Step 6b: Precision-Recall Curve & Threshold Tuning

The default threshold of 0.5 assumes equal costs for FP and FN.
For imbalanced data, tune the threshold by finding the operating point on the PR curve.

In [8]:
best_model = trained_models[best_model_name]
y_proba_best = best_model.predict_proba(X_test)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba_best)

# Find threshold closest to recall=0.7 (catch 70% of payers at some precision level)
idx_70 = np.argmin(np.abs(recalls - 0.7))
threshold_70_recall = thresholds[idx_70] if idx_70 < len(thresholds) else 0.5
precision_at_70 = precisions[idx_70]

# Find threshold closest to recall=0.8
idx_80 = np.argmin(np.abs(recalls - 0.8))
threshold_80_recall = thresholds[idx_80] if idx_80 < len(thresholds) else 0.5
precision_at_80 = precisions[idx_80]

print(f"=== Precision-Recall Threshold Tuning ({best_model_name}) ===")
print(f"At 70% recall: threshold={threshold_70_recall:.3f}, precision={precision_at_70:.3f}")
print(f"At 80% recall: threshold={threshold_80_recall:.3f}, precision={precision_at_80:.3f}")
print(f"Default (threshold=0.5) recall: {recalls[np.argmin(np.abs(thresholds - 0.5))]:.3f}")
print("\nPR curve gives you flexibility:")
print("  - High precision, low recall: catch fewer non-payers but confident")
print("  - Low precision, high recall: catch more non-payers but more false alarms")

=== Precision-Recall Threshold Tuning (XGBoost) ===
At 70% recall: threshold=0.910, precision=0.985
At 80% recall: threshold=0.802, precision=0.975
Default (threshold=0.5) recall: 0.909

PR curve gives you flexibility:
  - High precision, low recall: catch fewer non-payers but confident
  - Low precision, high recall: catch more non-payers but more false alarms


## Step 6c: Lift & Concentration Metrics

Rank predictions by score and ask: "What fraction of actual non-payers are in my top-K% of risk?"

A good model captures 70-80% of non-payers in the top 20% of scores. A random model captures 20%.

In [9]:
def compute_lift_metrics(y_true, y_pred_proba, percentiles=[10, 20, 30]):
    """
    Given predictions ranked by score, what % of positive class is captured
    in top K% of scores?

    Lift = (captured % / base %)
    Example: If 60% of minority class is in top 20%, lift = 60/20 = 3.0x
    """
    df_scores = pd.DataFrame({
        'y_true': y_true,
        'score': y_pred_proba
    })
    df_scores['rank_pct'] = pd.qcut(
        df_scores['score'], q=100, labels=False, duplicates='drop'
    ) / 100

    results = {}
    for pct in percentiles:
        top_k = df_scores[df_scores['rank_pct'] >= (1 - pct / 100)]
        captured_minority = top_k[top_k['y_true'] == 1].shape[0]
        total_minority = (df_scores['y_true'] == 1).sum()
        base_rate = pct / 100

        captured_pct = captured_minority / total_minority if total_minority > 0 else 0
        lift = captured_pct / base_rate if base_rate > 0 else 1.0

        results[f'top_{pct}pct'] = {
            'captured_minority': captured_minority,
            'captured_pct': captured_pct,
            'lift': lift
        }

    return results


lift = compute_lift_metrics(y_test.to_numpy(), y_proba_best, percentiles=[10, 20, 30])

print(f"=== Lift Metrics ({best_model_name}) ===")
print("(% of paid-within-30d invoices captured in top K% of scores)")
print()
for key, metrics in lift.items():
    print(f"{key}:")
    print(f"  Captured: {metrics['captured_pct']:.1%} of payers")
    print(f"  Lift: {metrics['lift']:.2f}x (1.0x = random, >1.5x = good)")
    print()

print("Random model would capture ~10% in top 10%, ~20% in top 20%, etc.")
print("A good model should have lift > 1.5 in top deciles.")

=== Lift Metrics (XGBoost) ===
(% of paid-within-30d invoices captured in top K% of scores)

top_10pct:
  Captured: 12.7% of payers
  Lift: 1.27x (1.0x = random, >1.5x = good)

top_20pct:
  Captured: 25.4% of payers
  Lift: 1.27x (1.0x = random, >1.5x = good)

top_30pct:
  Captured: 38.0% of payers
  Lift: 1.27x (1.0x = random, >1.5x = good)

Random model would capture ~10% in top 10%, ~20% in top 20%, etc.
A good model should have lift > 1.5 in top deciles.


## Step 6d: Synthetic Feature Impact Analysis

Trains a comparison model **without** synthetic appointment features to measure their impact.
This helps validate whether the high ROC-AUC is driven by real patterns or synthetic data artifacts.

**Key question:** If we remove `appointment_reliability_score`, how much does performance drop?

In [10]:
import json

# Only run comparison if synthetic appointments are currently included
if INCLUDE_SYNTHETIC_APPOINTMENTS:
    print('='*60)
    print(' SYNTHETIC FEATURE IMPACT ANALYSIS')
    print('='*60)
    
    # Define feature set without synthetic appointments
    features_no_synthetic = [f for f in feature_columns if f not in SYNTHETIC_APPOINTMENT_FEATURES]
    
    X_train_no_syn = X_train[features_no_synthetic]
    X_test_no_syn = X_test[features_no_synthetic]
    
    # Train comparison model
    print(f"\nTraining comparison model without synthetic appointment features...")
    print(f"  Features: {len(features_no_synthetic)} (removed: {SYNTHETIC_APPOINTMENT_FEATURES})")
    
    model_no_synthetic = ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('clf', XGBClassifier(
            n_estimators=200, random_state=42, eval_metric='logloss',
            scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum()
        ))
    ])
    
    model_no_synthetic.fit(X_train_no_syn, y_train)
    print("  Done.")
    
    # Evaluate
    y_pred_no_syn = model_no_synthetic.predict(X_test_no_syn)
    y_proba_no_syn = model_no_synthetic.predict_proba(X_test_no_syn)[:, 1]
    
    auc_no_syn = roc_auc_score(y_test, y_proba_no_syn)
    brier_no_syn = brier_score_loss(y_test, y_proba_no_syn)
    acc_no_syn = accuracy_score(y_test, y_pred_no_syn)
    
    # Get metrics from best model with synthetic features
    auc_with = results_df[results_df['Model'] == best_model_name]['ROC-AUC'].values[0]
    brier_with = results_df[results_df['Model'] == best_model_name]['Brier'].values[0]
    acc_with = results_df[results_df['Model'] == best_model_name]['Accuracy'].values[0]
    
    # Compare
    print(f"\n{'='*60}")
    print(' COMPARISON: WITH vs. WITHOUT SYNTHETIC APPOINTMENTS')
    print(f"{'='*60}")
    print(f"\n{'Metric':<20} {'With Synthetic':<20} {'Without Synthetic':<20} {'Delta':<15}")
    print(f"{'-'*75}")
    
    print(f"{'ROC-AUC':<20} {auc_with:<20.4f} {auc_no_syn:<20.4f} {auc_with - auc_no_syn:+.4f}")
    print(f"{'Brier Score':<20} {brier_with:<20.4f} {brier_no_syn:<20.4f} {brier_with - brier_no_syn:+.4f}")
    print(f"{'Accuracy':<20} {acc_with:<20.4f} {acc_no_syn:<20.4f} {acc_with - acc_no_syn:+.4f}")
    print(f"{'Features':<20} {len(feature_columns):<20} {len(features_no_synthetic):<20} {len(feature_columns) - len(features_no_synthetic):+d}")
    
    # Interpretation
    print(f"\n{'='*60}")
    print(' INTERPRETATION')
    print(f"{'='*60}")
    
    auc_delta = auc_with - auc_no_syn
    if abs(auc_delta) < 0.01:
        impact = 'NEGLIGIBLE'
        recommendation = 'Synthetic appointment feature adds no value. Safe to remove.'
    elif auc_delta > 0.05:
        impact = 'SIGNIFICANT POSITIVE'
        recommendation = 'Synthetic feature is contributing. Investigate why before using in production.'
    elif auc_delta < -0.05:
        impact = 'SIGNIFICANT NEGATIVE'
        recommendation = 'Synthetic feature is hurting performance. Remove immediately.'
    else:
        impact = 'MODEST'
        recommendation = 'Synthetic feature has small impact. Consider removing for simplicity.'
    
    print(f"\nImpact: {impact}")
    print(f"AUC Delta: {auc_delta:+.4f}")
    print(f"Recommendation: {recommendation}")
    
    # Save comparison results
    comparison_results = {
        'with_synthetic': {
            'features': len(feature_columns),
            'feature_list': feature_columns,
            'roc_auc': float(auc_with),
            'brier': float(brier_with),
            'accuracy': float(acc_with),
        },
        'without_synthetic': {
            'features': len(features_no_synthetic),
            'feature_list': features_no_synthetic,
            'roc_auc': float(auc_no_syn),
            'brier': float(brier_no_syn),
            'accuracy': float(acc_no_syn),
        },
        'delta': {
            'roc_auc': float(auc_delta),
            'brier': float(brier_with - brier_no_syn),
            'accuracy': float(acc_with - acc_no_syn),
        },
        'impact_assessment': impact,
        'recommendation': recommendation,
        'synthetic_features_tested': SYNTHETIC_APPOINTMENT_FEATURES,
    }
    
    comparison_path = '../../models/synthetic_feature_impact_analysis.json'
    with open(comparison_path, 'w') as f:
        json.dump(comparison_results, f, indent=2)
    print(f"\n✅ Saved comparison results to {comparison_path}")

else:
    print('='*60)
    print(' SYNTHETIC FEATURE IMPACT ANALYSIS')
    print('='*60)
    print('\n⚠️  Synthetic appointment features are EXCLUDED in this run.')
    print('    Set INCLUDE_SYNTHETIC_APPOINTMENTS = True to run comparison.')

 SYNTHETIC FEATURE IMPACT ANALYSIS

Training comparison model without synthetic appointment features...
  Features: 16 (removed: ['appointment_reliability_score'])
  Done.

 COMPARISON: WITH vs. WITHOUT SYNTHETIC APPOINTMENTS

Metric               With Synthetic       Without Synthetic    Delta          
---------------------------------------------------------------------------
ROC-AUC              0.9469               0.9460               +0.0009
Brier Score          0.0778               0.0776               +0.0001
Accuracy             0.8926               0.8916               +0.0010
Features             17                   16                   +1

 INTERPRETATION

Impact: NEGLIGIBLE
AUC Delta: +0.0009
Recommendation: Synthetic appointment feature adds no value. Safe to remove.

✅ Saved comparison results to ../../models/synthetic_feature_impact_analysis.json


## Step 7: Feature Importance (Best Model)

In [11]:
# Extract the underlying estimator from the imblearn Pipeline
best_estimator = best_model.named_steps['clf']

if hasattr(best_estimator, 'feature_importances_'):
    importances = best_estimator.feature_importances_
elif hasattr(best_estimator, 'coef_'):
    importances = np.abs(best_estimator.coef_[0])
else:
    importances = None

if importances is not None:
    importance_df = pd.DataFrame({
        'Feature': feature_columns,
        'Importance': importances
    }).sort_values('Importance', ascending=False)

    print(f"Feature importance ({best_model_name}):")
    print(importance_df.to_string(index=False))
else:
    print("Feature importance not available for this model type.")

Feature importance (XGBoost):
                      Feature  Importance
          payment_origin_mode    0.533211
                tenure_months    0.172524
          average_days_to_pay    0.053429
               is_new_patient    0.031025
         created_day_of_month    0.028256
             account_age_days    0.024843
          created_day_of_week    0.023261
                 created_hour    0.019922
                       amount    0.016891
      median_household_income    0.015243
appointment_reliability_score    0.014644
             poverty_rate_pct    0.014108
        unemployment_rate_pct    0.013789
                  is_guardian    0.013781
       average_household_size    0.012637
      bachelors_or_higher_pct    0.012436
           surchargingenabled    0.000000


## Step 8: Save Model Artifact

In [12]:
import joblib
import json
import os

artifacts_dir = '../../models'
os.makedirs(artifacts_dir, exist_ok=True)

# Save model
model_path = os.path.join(artifacts_dir, 'will_they_pay_prod_v1.joblib')
joblib.dump(best_model, model_path)
print(f"\u2705 Saved model: {model_path}")

# Save feature columns (required by batch scorer per TPD)
features_path = os.path.join(artifacts_dir, 'feature_columns_prod.json')
with open(features_path, 'w') as f:
    json.dump(feature_columns, f, indent=2)
print(f"\u2705 Saved feature columns: {features_path}")

# Save evaluation summary
eval_summary = {
    'best_model': best_model_name,
    'roc_auc': float(results_df.iloc[0]['ROC-AUC']),
    'brier_score': float(results_df.iloc[0]['Brier']),
    'training_rows': int(X_train.shape[0]),
    'test_rows': int(X_test.shape[0]),
    'features': feature_columns,
    'paid_rate': float(paid_rate),
    'uses_smote': True,
    'synthetic_appointments_included': INCLUDE_SYNTHETIC_APPOINTMENTS,
    'synthetic_person_features_included': INCLUDE_SYNTHETIC_PERSON_FEATURES,
}
eval_path = os.path.join(artifacts_dir, 'eval_summary_prod.json')
with open(eval_path, 'w') as f:
    json.dump(eval_summary, f, indent=2)
print(f"\u2705 Saved evaluation summary: {eval_path}")

✅ Saved model: ../../models/will_they_pay_prod_v1.joblib
✅ Saved feature columns: ../../models/feature_columns_prod.json
✅ Saved evaluation summary: ../../models/eval_summary_prod.json


## Step 9: Batch Score Open Invoices

Scores currently open (unpaid) invoices with the trained model.
Outputs prediction + confidence + risk band per the TPD schema.

Risk bands:
- `>= 0.70` → **high** (likely to pay)
- `0.40 – 0.69` → **medium**
- `< 0.40` → **low** (unlikely to pay)

In [13]:
# Filter to open invoices only.
# Status codes per payments.InvoiceStatus enum:
#   5 = PARTIALLY_PAID
#   6 = UNPAID
#   7 = SCHEDULED
#   9 = INVOICE_STATUS_PENDING
# See docs/db-schemas/payments/status-enums.md for full enum.
open_statuses = [5, 6, 7, 9]  # PARTIALLY_PAID, UNPAID, SCHEDULED, INVOICE_STATUS_PENDING
open_invoices = training_df[training_df['status'].isin(open_statuses)].copy()

print(f"Open invoices to score: {len(open_invoices):,}")

if len(open_invoices) > 0:
    # Extract features
    X_open = open_invoices[feature_columns]

    # Predict
    open_invoices['will_pay_in_30'] = best_model.predict(X_open)
    open_invoices['confidence_score'] = best_model.predict_proba(X_open)[:, 1]

    # Derive risk band per TPD
    open_invoices['risk_band'] = pd.cut(
        open_invoices['confidence_score'],
        bins=[0, 0.40, 0.70, 1.0],
        labels=['low', 'medium', 'high'],
        include_lowest=True
    )

    print(f"\nRisk band distribution:")
    print(open_invoices['risk_band'].value_counts().to_string())

    print(f"\nConfidence score stats:")
    print(f"   Mean:   {open_invoices['confidence_score'].mean():.3f}")
    print(f"   Median: {open_invoices['confidence_score'].median():.3f}")
    print(f"   Min:    {open_invoices['confidence_score'].min():.3f}")

    print(f"   Max:    {open_invoices['confidence_score'].max():.3f}")

    print(f"\nPreview:")
    display(open_invoices[['personid', 'amount', 'status', 'will_pay_in_30', 'confidence_score', 'risk_band']].head(10))

else:
    print("No open invoices found in the training dataset.")

Open invoices to score: 5,210

Risk band distribution:
risk_band
low       4871
high       171
medium     168

Confidence score stats:
   Mean:   0.104
   Median: 0.030
   Min:    0.000
   Max:    1.000

Preview:


,personid,amount,status,will_pay_in_30,confidence_score,risk_band
3,11af2c63-9a57-556a-91e7-c157840adf45,106520,6,0,0.097877,low
4,11af2c63-9a57-556a-91e7-c157840adf45,19000,6,0,0.069891,low
6,11af2c63-9a57-556a-91e7-c157840adf45,55600,5,0,0.048248,low
7,11af2c63-9a57-556a-91e7-c157840adf45,46284,5,0,0.454934,medium
10,12314eca-85fb-575f-ab61-9f6e2ba73a1e,13142,6,1,0.500577,medium
11,13b1a799-184a-5751-852c-51259d40454d,14910,6,1,0.647246,medium
15,14f2a261-89a6-5e9b-a9a2-a9a2876d1ed0,14660,6,0,0.000068,low
16,14f2a261-89a6-5e9b-a9a2-a9a2876d1ed0,14660,6,0,0.000262,low
17,14f2a261-89a6-5e9b-a9a2-a9a2876d1ed0,14660,6,0,0.000070,low
18,14f2a261-89a6-5e9b-a9a2-a9a2876d1ed0,14880,6,0,0.000169,low


## Step 10: Save Scored Predictions

In [14]:
from datetime import datetime, timezone

if len(open_invoices) > 0:
    predictions_output = open_invoices[[
        'personid', 'postal_code', 'will_pay_in_30', 'confidence_score', 'risk_band'
    ]].copy()
    predictions_output['model_version'] = 'will_they_pay_prod_v1'
    predictions_output['scored_at'] = datetime.now(timezone.utc).isoformat()

    output_path = '../../data/output/scored_predictions_prod.csv'
    predictions_output.to_csv(output_path, index=False)
    print(f"\u2705 Saved {len(predictions_output):,} scored predictions to {output_path}")
    predictions_output.head()
else:
    print("No predictions to save (no open invoices).")

✅ Saved 5,210 scored predictions to ../../data/output/scored_predictions_prod.csv
